# Recipe 2 — Extract Overlaps from Specific Tracks

This notebook walks through querying the FILER overlaps endpoint to pull the actual genomic
features (peaks, intervals, etc.) from a set of tracks that fall within a region of interest.

**What you'll learn:**
- How to query the FILER overlaps endpoint with a region and track ID list
- How to load track IDs directly or from a Recipe 1 `tracks.tsv` / `.trackset.json`
- How to explore and summarize the returned intervals
- How to save results as TSV and JSON for downstream workflows

**Use when:** you have a small-to-medium set of tracks from Recipe 1 and want the actual
intervals overlapping a locus. Expect ~1–2 seconds per batch of 250 tracks.

**Next steps after this notebook:**
- `01_track_discovery.ipynb` — find and filter tracks before querying
- `03_coordinate_search.ipynb` — discover tracks by region without a pre-selected ID list

Run the following before starting:
```
pip install -e ".[dev]"
```

---
## 0. Setup

In [ ]:
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from filerpy.client import fetch_overlaps, load_ids_from_file, ENDPOINTS

In [ ]:
print(f"FILER overlaps endpoint: {ENDPOINTS['overlaps']}")

---
## 1. Hello World — query a region against two known tracks

The fastest way to verify your connection. `fetch_overlaps` handles batching and
error checking automatically and returns a flat DataFrame.

In [ ]:
df_hello = fetch_overlaps(
    region="chr19:44905791-44909393",
    ids=["NGBLPL2W2SM2WC", "NGENCC4CIT5FBQ"],
)

In [ ]:
print(f"Intervals returned: {len(df_hello)}")
print(df_hello.head().to_string())

---
## 2. Configure your query

Set your region and choose how to supply track IDs.

| Source | When to use |
|---|---|
| `TRACK_IDS` list | A handful of known IDs |
| `TRACKS_FILE` path | Output from Recipe 1 — recommended for larger searches |

If both are set, `TRACK_IDS` takes precedence.

In [ ]:
# ── Edit these parameters to match your use case ───────────────────────────
REGION     = "chr19:44905791-44909393"             # required; format: chrN:start-end

In [ ]:
# Option A: supply IDs directly (takes precedence over TRACKS_FILE)
TRACK_IDS  = ["NGBLPL2W2SM2WC", "NGENCC4CIT5FBQ"]  # set to [] to use TRACKS_FILE
TRACK_IDS  = []

In [ ]:
# Option B: load IDs from a file — see section 2b
TRACKS_FILE = "../../output/01-track-discovery/tracks.tsv"          # e.g. "output/01-track-discovery/recipe01_tracks.tsv"
ID_COL      = "identifier"  # column name in the TSV; ignored for .json files

In [ ]:
CHUNK_SIZE = 250    # IDs per request; lower to 100 if the server returns errors
SAVE_JSON  = True   # also write overlaps.json alongside overlaps.tsv
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
if TRACK_IDS:
    track_ids = list(TRACK_IDS)
    print(f"Using {len(track_ids)} IDs supplied directly")
elif TRACKS_FILE:
    print(f"TRACKS_FILE set — continue to section 2b to load IDs from: {TRACKS_FILE}")
    track_ids = []
else:
    print("No IDs configured — set TRACK_IDS or TRACKS_FILE above")
    track_ids = []

### 2b. Load track IDs from a file (recommended for large searches)

Uses `load_ids_from_file` from `filer_find_overlaps.py` — the same logic as the `--file` +
`--id-col` flags in the CLI script.

| File type | How IDs are extracted |
|---|---|
| `.tsv` | Read `ID_COL` column, drop nulls, deduplicate |
| `.json` | Parse as a JSON array of records, read `ID_COL` field |

> ℹ️ Skipped automatically when `TRACK_IDS` is non-empty.

In [ ]:
if TRACK_IDS:
    print(f"TRACK_IDS is set — skipping file load ({len(track_ids)} direct IDs)")
elif TRACKS_FILE:
    # load_ids_from_file mirrors --file + --id-col from the CLI script.
    # It validates that the file exists and that ID_COL is present, and exits clearly on errors.
    track_ids = load_ids_from_file(TRACKS_FILE, ID_COL)
    print(f"Loaded {len(track_ids)} unique IDs from {TRACKS_FILE}")
    print(f"First 5 IDs: {track_ids[:5]}")
else:
    raise ValueError("Set either TRACK_IDS or TRACKS_FILE in section 2 before continuing")

In [ ]:
print(f"\nRegion  : {REGION}")
print(f"Tracks  : {len(track_ids)}")
print(f"Batches : {-(-len(track_ids) // CHUNK_SIZE)} x {CHUNK_SIZE}")

---
## 3. Run the query

`fetch_overlaps` splits the ID list into batches of `CHUNK_SIZE`, issues one GET request
per batch, and concatenates the results into a single DataFrame.

> ⚠️ Expect roughly 1 minute per batch of 250 tracks

In [ ]:
df = fetch_overlaps(REGION, track_ids, chunk_size=CHUNK_SIZE)

In [ ]:
print(f"\nTotal overlapping intervals: {len(df)}")
print(df.head().to_string())

### 3a. View key columns only

In [ ]:
PREFERRED_COLS = ["Identifier", "queryRegion", "hitString"]
KEY_COLS = [c for c in PREFERRED_COLS if c in df.columns]

In [ ]:
print(f"Displaying columns: {KEY_COLS}")
print(df[KEY_COLS].head(10).to_string())

---
## 4. Explore the results

In [ ]:
print("=== Intervals per track (top 20) ===")
print(df["Identifier"].value_counts().head(20).to_string())

In [ ]:
if "hitString" in df.columns:
    hit_lens = df["hitString"].astype(str).str.split("@@@" ).str.len()
    print("=== hitString token count distribution ===")
    print(hit_lens.describe())

In [ ]:
# Bar chart — intervals per track (top 20)
counts = df["Identifier"].value_counts().head(20)
fig, ax = plt.subplots(figsize=(8, max(3, len(counts) * 0.4)))
counts.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Number of overlapping intervals")
ax.set_title(f"Overlapping intervals per track\n({REGION})")
plt.tight_layout()
plt.show()

In [ ]:
# Histogram — hitString token counts
if "hitString" in df.columns:
    hit_lens = df["hitString"].astype(str).str.split("@@@").str.len()
    fig, ax = plt.subplots(figsize=(8, 4))
    hit_lens.plot.hist(ax=ax, bins=40, color="orchid", edgecolor="white")
    ax.set_xlabel("Number of fields in hitString")
    ax.set_ylabel("Count")
    ax.set_title(f"Distribution of hitString field counts\n({REGION})")
    plt.tight_layout()
    plt.show()

---
## 5. Refine your selection

Post-query filtering in pandas — useful for narrowing by hitString contents (or token counts) and specific tracks.

In [ ]:
df_filtered = df.copy()

In [ ]:
# ── Edit these conditions to match what you need ────────────────────────────
if "hitString" in df_filtered.columns:
    hit_lens = df_filtered["hitString"].astype(str).str.split("@@@").str.len()
    df_filtered = df_filtered[hit_lens >= 3]

In [ ]:
if "hitString" in df_filtered.columns and df_filtered["hitString"].astype(str).str.contains("Peak", na=False).any():
    df_filtered = df_filtered[df_filtered["hitString"].astype(str).str.contains("Peak", na=False)]
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
print(f"After refinement: {len(df_filtered)} intervals (from {len(df)} total)")
print(df_filtered[KEY_COLS].head().to_string())

---
## 6. Save results

### 6a. Save as TSV

In [ ]:
repo_root = Path().resolve().parents[1]
out_dir = repo_root / "output" / "02-track-overlaps"
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
tsv_path = out_dir / "overlaps.tsv"
df_filtered.to_csv(tsv_path, sep="\t", index=False)
print(f"Saved {len(df_filtered)} intervals -> {tsv_path}")

### 6b. Save as JSON (optional)

In [ ]:
if SAVE_JSON:
    json_path = out_dir / "overlaps.json"
    df_filtered.to_json(json_path, orient="records", indent=2)
    print(f"Saved JSON -> {json_path}")
else:
    print("SAVE_JSON=False — skipping JSON output")

### 6c. Save query provenance (recommended for reproducibility)

In [ ]:
provenance = {
    "query": {
        "region":     REGION,
        "track_ids":  track_ids,
        "chunk_size": CHUNK_SIZE,
        "source":     TRACKS_FILE if TRACKS_FILE else "direct",
    },
    "results": {
        "total_intervals":  len(df),
        "after_refinement": len(df_filtered),
    },
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

In [ ]:
prov_path = out_dir / "overlaps_provenance.json"
prov_path.write_text(json.dumps(provenance, indent=2))
print(f"Provenance saved -> {prov_path}")
print(json.dumps(provenance, indent=2))

---
## 7. Next steps

You now have `overlaps.tsv`, `overlaps.json`, and `overlaps_provenance.json` ready for
downstream use.

| What to do next | Recipe / Notebook |
|---|---|
| Find and filter tracks before querying | Recipe 1 / `01_track_discovery.ipynb` |
| Discover tracks by region without a pre-selected ID list | Recipe 3 / `03_coordinate_search.ipynb` |
| Summarize overlaps by assay / cell type | Recipe 4.1 |
| Deploy tracks locally for fast offline queries | Recipe 2.2 |

```python
# Quick reference: reload your saved results in a future session
import pandas as pd
df = pd.read_csv("output/02-track-overlaps/overlaps.tsv", sep="\t")
print(df.head())
```